In [1]:
# Core imports
import sys
import json
import os
from pathlib import Path

import pandas as pd

_here = Path.cwd()
if (_here / "utils").exists():
    pass
elif (_here / "notebooks" / "utils").exists():
    sys.path.insert(0, str(_here / "notebooks"))

import dspy
from typing import List, Optional, Literal, Union
from pydantic import BaseModel, Field
from tqdm import tqdm


In [2]:
# DSPy Configuration
lm = dspy.LM(
    model="openai/gpt-5.2", 
    api_key=os.environ["OPENAI_API_KEY"],
)
dspy.configure(lm=lm)



### Core Policy Model

In [3]:
from utils.schemas import ExtractedPolicy

In [4]:
from utils.schemas import DocumentMetadata 


---

## 🧠 DSPy Signatures & Modules {#dspy}

Define DSPy signatures and custom modules for policy extraction, validation, and processing.

### Policy Extraction Signature

In [5]:
from utils.dspy_extraction import PolicyExtractionSignature, PolicyExtractor

In [6]:
# (Legacy strict validator removed)
# Individual validation uses the refined validator in `utils.dspy_validation`.
from utils.dspy_validation import PolicyValidationSignature, PolicyValidator, ValidationMetrics

In [7]:
from utils.docling_io import pdf_to_markdown

---

## 🗂️ Batch Configuration

Define the list of cities to process. Add a `DocumentMetadata` + PDF path per city. All 7 steps below will loop over this list automatically.

In [8]:
# ─── Batch Configuration ────────────────────────────────────────────────────
# Each entry has:
#   metadata     -> DocumentMetadata (country, state_or_province, city)
#   pdf_path     -> original PDF (only used if markdown_path is missing)
#   markdown_path -> pre-converted markdown file (skips PDF conversion entirely)
#
# Outputs are written to OUTPUT_DIR/<city_key>/

from utils.schemas import DocumentMetadata

BATCH = [
    {
        "metadata": DocumentMetadata(country="United States", state_or_province="Illinois", city="Chicago"),
        "pdf_path": "../pdfs/Chicago-CAP-071822.pdf",
        "markdown_path": "../docs/cities/chicago.md",
    },
    {
        "metadata": DocumentMetadata(country="United States", state_or_province="Washington", city="Seattle"),
        "pdf_path": "../pdfs/Seattle.pdf",
        "markdown_path": "../docs/cities/seattle_markdown.md",
    },
    {
        "metadata": DocumentMetadata(country="United States", state_or_province="Nevada", city="Las Vegas"),
        "pdf_path": "../pdfs/CLV.pdf",
        "markdown_path": "../docs/cities/LV.md",
    },
    {
        "metadata": DocumentMetadata(country="United States", state_or_province="Florida", city="Miami-Dade"),
        "pdf_path": "../pdfs/MIAMI_DADE.pdf",
        "markdown_path": "../docs/cities/miami_markdown.md",
    },
    {
        "metadata": DocumentMetadata(country="United States", state_or_province="Texas", city="Austin"),
        "pdf_path": "../pdfs/Austin_climate_equity.pdf",
        "markdown_path": "../docs/cities/austin.md",
    },
    {
        "metadata": DocumentMetadata(country="Senegal", city="Dakar"),
        "pdf_path": "../pdfs/Dakar.pdf",
        "markdown_path": "../docs/cities/dakar.md",
    },
    {
        "metadata": DocumentMetadata(country="Kuwait"),
        "pdf_path": "../pdfs/Kuwait-NAP-2019-2030.pdf",
        "markdown_path": "../docs/cities/kuwait.md",
    },
    {
        "metadata": DocumentMetadata(country="Portugal"),
        "pdf_path": "../pdfs/2021 Portugal ADCOM_UNFCCC.pdf",
        "markdown_path": "../docs/cities/Portugal.md",
    },
    {
        "metadata": DocumentMetadata(country = "Switzerland",city= "Geneva"),
        "pdf_path":"../pdfs/city_of_geneva.pdf",
        "markdown_path":"../docs/cities/geneva.md",

    },
    {
        "metadata":DocumentMetadata(country="Japan",city="Hiroshima"),
        "pdf_path":"../pdfs/HIROSHIMA.pdf",
        "markdown_path":"../docs/cities/Hiroshima.md",
    }
]

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

# Number of parallel threads for LLM calls (Steps 5 & 6).
NUM_THREADS = 8

def city_key(metadata: DocumentMetadata) -> str:
    """Slug used as filename prefix and output folder name."""
    parts = [metadata.city or metadata.country, metadata.country]
    # e.g. 'Chicago_United_States' or 'Kuwait_Kuwait' → just use city if present
    label = metadata.city if metadata.city else metadata.country
    return label.replace(" ", "_").replace("-", "_")

print(f"Batch configured ({len(BATCH)} cities):")
for e in BATCH:
    key = city_key(e["metadata"])
    md = e.get("markdown_path", "—")
    print(f"  {key:20s}  md={md}")
print(f"\nParallel threads: {NUM_THREADS}")


Batch configured (10 cities):
  Chicago               md=../docs/cities/chicago.md
  Seattle               md=../docs/cities/seattle_markdown.md
  Las_Vegas             md=../docs/cities/LV.md
  Miami_Dade            md=../docs/cities/miami_markdown.md
  Austin                md=../docs/cities/austin.md
  Dakar                 md=../docs/cities/dakar.md
  Kuwait                md=../docs/cities/kuwait.md
  Portugal              md=../docs/cities/Portugal.md
  Geneva                md=../docs/cities/geneva.md
  Hiroshima             md=../docs/cities/Hiroshima.md

Parallel threads: 8


### Step 1 — Load Markdown

Loads each city's document as markdown text using this priority order:
1. **Pre-converted file** supplied via `markdown_path` in `BATCH` (e.g. `docs/cities/*.md`) — instant, no API call
2. **Cached conversion** saved by a previous run (`outputs/{key}_markdown.md`)
3. **Live PDF conversion** via Docling — only runs if neither of the above exists, and caches the result


In [9]:
# Step 1: Load Markdown (pre-converted files take priority; falls back to PDF conversion)
from utils.docling_io import pdf_to_markdown

markdowns = {}  # city_key -> markdown string

for entry in BATCH:
    key = city_key(entry["metadata"])
    md_path = entry.get("markdown_path")

    # ── Priority 1: pre-existing markdown file supplied in config ────────
    if md_path and Path(md_path).exists():
        with open(md_path, "r", encoding="utf-8") as f:
            markdowns[key] = f.read()
        print(f"[{key}] Loaded pre-converted markdown from {md_path}  ({len(markdowns[key]):,} chars)")

    # ── Priority 2: cached conversion saved by a previous run ────────────
    elif (OUTPUT_DIR / f"{key}_markdown.md").exists():
        cached = OUTPUT_DIR / f"{key}_markdown.md"
        with open(cached, "r", encoding="utf-8") as f:
            markdowns[key] = f.read()
        print(f"[{key}] Loaded cached markdown from {cached}  ({len(markdowns[key]):,} chars)")

    # ── Priority 3: convert from PDF and cache the result ────────────────
    else:
        pdf = entry.get("pdf_path")
        if not pdf or not Path(pdf).exists():
            print(f"[{key}]  No markdown and no PDF found — skipping.")
            continue
        cache_path = OUTPUT_DIR / f"{key}_markdown.md"
        print(f"[{key}] Converting PDF: {pdf} ...")
        markdowns[key] = pdf_to_markdown(pdf, save_markdown_path=str(cache_path))
        print(f"[{key}] Done  ({len(markdowns[key]):,} chars)  → cached at {cache_path}")


[Chicago] Loaded pre-converted markdown from ../docs/cities/chicago.md  (256,905 chars)
[Seattle] Loaded pre-converted markdown from ../docs/cities/seattle_markdown.md  (49,888 chars)
[Las_Vegas] Loaded pre-converted markdown from ../docs/cities/LV.md  (61,953 chars)
[Miami_Dade] Loaded pre-converted markdown from ../docs/cities/miami_markdown.md  (141,191 chars)
[Austin] Loaded pre-converted markdown from ../docs/cities/austin.md  (413,340 chars)
[Dakar] Loaded pre-converted markdown from ../docs/cities/dakar.md  (227,420 chars)
[Kuwait] Loaded pre-converted markdown from ../docs/cities/kuwait.md  (362,332 chars)
[Portugal] Loaded pre-converted markdown from ../docs/cities/Portugal.md  (438,819 chars)
[Geneva] Loaded pre-converted markdown from ../docs/cities/geneva.md  (19,174 chars)
[Hiroshima] Loaded pre-converted markdown from ../docs/cities/Hiroshima.md  (87,414 chars)


### Step 2 — Extract Policies

Runs the DSPy `PolicyExtractor` LLM call for each city. Saves `{City_Country}_extracted_policies.json` to `outputs/`.

In [10]:
# Step 2: Extract Policies — one LLM call per city, parallelized across cities
from utils.dspy_extraction import PolicyExtractor
from dspy.utils.parallelizer import ParallelExecutor

policy_extractor = PolicyExtractor()
all_extracted = {}  # city_key -> List[ExtractedPolicy]

# Only extract for cities whose markdown was successfully loaded
batch_to_extract = [e for e in BATCH if city_key(e["metadata"]) in markdowns]

def extract_one(entry):
    key = city_key(entry["metadata"])
    extracted = policy_extractor(
        document_text=markdowns[key],
        document_metadata=entry["metadata"],
    )
    return {"key": key, "extracted": extracted}

print(f"Extracting policies for {len(batch_to_extract)} cities across {NUM_THREADS} threads...\n")

executor = ParallelExecutor(
    num_threads=min(NUM_THREADS, len(batch_to_extract)),
    max_errors=len(batch_to_extract),
    provide_traceback=True,
)
raw_results = executor.execute(extract_one, batch_to_extract)

# ── Collect results in memory, then write — no concurrent file I/O ──────────
n_errors = 0
for i, res in enumerate(raw_results):
    if res is None or isinstance(res, Exception):
        n_errors += 1
        key = city_key(batch_to_extract[i]["metadata"])
        print(f"  ⚠ [{key}] extraction failed: {res}")
    else:
        key = res["key"]
        all_extracted[key] = res["extracted"]

# ── Write all JSON files after all threads are done ──────────────────────────
for key, extracted in all_extracted.items():
    out = OUTPUT_DIR / f"{key}_extracted_policies.json"
    with open(out, "w", encoding="utf-8") as f:
        json.dump([p.model_dump() for p in extracted], f, ensure_ascii=False, indent=2)
    print(f"[{key}] {len(extracted)} policies → {out}")

print(f"\nDone. {len(all_extracted)} succeeded, {n_errors} failed.")


Extracting policies for 10 cities across 8 threads...

Processed 10 / 10 examples: 100%|██████████| 10/10 [00:00<00:00, 27.12it/s]
[Chicago] 68 policies → outputs/Chicago_extracted_policies.json
[Seattle] 27 policies → outputs/Seattle_extracted_policies.json
[Las_Vegas] 22 policies → outputs/Las_Vegas_extracted_policies.json
[Miami_Dade] 39 policies → outputs/Miami_Dade_extracted_policies.json
[Austin] 109 policies → outputs/Austin_extracted_policies.json
[Dakar] 17 policies → outputs/Dakar_extracted_policies.json
[Kuwait] 45 policies → outputs/Kuwait_extracted_policies.json
[Portugal] 22 policies → outputs/Portugal_extracted_policies.json
[Geneva] 12 policies → outputs/Geneva_extracted_policies.json
[Hiroshima] 46 policies → outputs/Hiroshima_extracted_policies.json

Done. 10 succeeded, 0 failed.


### Step 3 — Cluster Policies

Groups extracted policies by section header into `parent_with_subs`, `individual`, and `orphan_sub` clusters. Saves `{City_Country}_policy_clusters.json`.

In [11]:
# Step 3: Cluster Policies
from utils.clustering import cluster_policies, summarize_clusters

all_clusters = {}  # city_key -> List[dict]

for entry in BATCH:
    key = city_key(entry["metadata"])
    clusters = cluster_policies(all_extracted[key])
    all_clusters[key] = clusters

    print(f"\n[{key}]")
    summarize_clusters(clusters)

    out = OUTPUT_DIR / f"{key}_policy_clusters.json"
    with open(out, "w", encoding="utf-8") as f:
        json.dump(clusters, f, default=lambda o: o.model_dump(), ensure_ascii=False, indent=2)
    print(f"  → {out}")



[Chicago]
Total clusters:        17
  Parent+sub clusters: 14
  Individual clusters: 3
  Orphaned subs:       0

[Parent 1] [S T R A T E G Y 1 | RETROFIT BUILDINGS]
  Retrofit buildings to reduce energy use and GHG emissions across residential, mu
  └─ Sub 1: Retrofit residential buildings with 4 or fewer units: 20% by 2030 and 
  └─ Sub 2: Retrofit 20% of total 5+ unit residential buildings by 2030.
  └─ Sub 3: Retrofit 20% of total industrial buildings by 2030.
  └─ Sub 4: Retrofit 90% of total city-owned and sister agency-owned buildings by 
  └─ Sub 5: Retrofit 20% of total commercial buildings by 2035.
[Parent 2] [S T R A T E G Y 2 | CONNECT COMMUNITIES TO RENEWABLE ENERGY]
  Connect communities to renewable energy through community ownership models and e
  └─ Sub 1: Install 5 megawatts of co-owned community solar projects by 2025.
  └─ Sub 2: Increase Chicago-based community renewables to 20 megawatts by 2025.
  └─ Sub 3: Increase community renewables subscriptions so that low-i

### Step 4 — Build Policy Records

Flattens each city's clusters into a structured DataFrame with consistent columns. Saves `{City_Country}_policy_records.csv`.

In [14]:
# Step 4: Build Policy Records DataFrames
from utils.clustering import clusters_to_records

FRONT_COLS = [
    "cluster_id", "cluster_type", "role", "section_header", "sector",
    "policy_statement", "parent_statement", "verbatim_text", "extraction_rationale",
]

all_policy_records = {}  # city_key -> List[dict]
all_df_policies = {}     # city_key -> pd.DataFrame

for entry in BATCH:
    key = city_key(entry["metadata"])
    records = clusters_to_records(all_clusters[key])
    df = pd.DataFrame(records)
    df = df[FRONT_COLS + [c for c in df.columns if c not in FRONT_COLS]]

    all_policy_records[key] = records
    all_df_policies[key] = df

    # out = OUTPUT_DIR / f"{key}_policy_records.csv"
    # df.to_csv(out, index=False)
    # print(f"[{key}] {len(df)} records → {out}")


### Step 5 — Validate Individual Policies

Runs `PolicyValidator` on each `individual`-role policy. Flattens Pydantic metrics into columns. Saves `{City_Country}_validation_individual.csv`.

In [ ]:
# Step 5: Validate Individual Policies — parallel LLM calls via DSPy ParallelExecutor
from utils.dspy_validation import PolicyValidator
from dspy.utils.parallelizer import ParallelExecutor

validator = PolicyValidator()
all_df_final = {}  # city_key -> pd.DataFrame (flattened metrics)

for entry in BATCH:
    key = city_key(entry["metadata"])
    individual = [r for r in all_policy_records[key] if r.get("role") == "individual"]
    print(f"\n[{key}] Validating {len(individual)} individual policies "
          f"across {NUM_THREADS} threads...")

    # ── Build per-record task function ──────────────────────────────────
    def validate_one(record):
        pred = validator(policy_data=record)
        return {**record, **pred.toDict()}

    # ── Execute in parallel — results land in-order, errors → None ──────
    executor = ParallelExecutor(
        num_threads=NUM_THREADS,
        max_errors=len(individual),   # don't abort on individual failures
        provide_traceback=True,
    )
    raw_results = executor.execute(validate_one, individual)

    # ── Filter out None (failed calls) and collect errors ────────────────
    validated = []
    n_errors = 0
    for i, res in enumerate(raw_results):
        if res is None or isinstance(res, Exception):
            n_errors += 1
            print(f"  ⚠ Record {i} failed: {res}")
        else:
            validated.append(res)

    print(f"[{key}] {len(validated)} succeeded, {n_errors} failed")

    # ── Build DataFrame (all results now in memory) ───────────────────────
    df_val = pd.DataFrame(validated)

    # Flatten Pydantic validation_results column into separate columns
    if "validation_results" in df_val.columns:
        vdata = df_val["validation_results"].apply(
            lambda x: x.model_dump() if hasattr(x, "model_dump") else x.dict()
        )
        df_metrics = pd.json_normalize(vdata)
        df_val = pd.concat([df_val.drop(columns=["validation_results"]), df_metrics], axis=1)

    all_df_final[key] = df_val

    # ── Write once, after all results are collected ───────────────────────
    # out = OUTPUT_DIR / f"{key}_validation_individual.csv"
    # df_val.to_csv(out, index=False)
    # print(f"[{key}] Saved → {out}")



[Chicago] Validating 3 individual policies across 8 threads...
Processed 3 / 3 examples: 100%|██████████| 3/3 [00:00<00:00, 97.31it/s]
[Chicago] 3 succeeded, 0 failed
[Chicago] Saved → outputs/Chicago_validation_individual.csv

[Seattle] Validating 14 individual policies across 8 threads...
Processed 14 / 14 examples: 100%|██████████| 14/14 [00:00<00:00, 121.15it/s]
[Seattle] 14 succeeded, 0 failed
[Seattle] Saved → outputs/Seattle_validation_individual.csv

[Las_Vegas] Validating 7 individual policies across 8 threads...
Processed 7 / 7 examples: 100%|██████████| 7/7 [00:00<00:00, 220.38it/s]
[Las_Vegas] 7 succeeded, 0 failed
[Las_Vegas] Saved → outputs/Las_Vegas_validation_individual.csv

[Miami_Dade] Validating 12 individual policies across 8 threads...
Processed 12 / 12 examples: 100%|██████████| 12/12 [00:00<00:00, 131.73it/s]
[Miami_Dade] 12 succeeded, 0 failed
[Miami_Dade] Saved → outputs/Miami_Dade_validation_individual.csv

[Austin] Validating 13 individual policies across 8 

### Step 6 — Validate Initiatives

Runs `run_initiative_validation` on each city's `parent_with_subs` clusters. Saves `{City_Country}_validated_initiatives.csv`. 


In [14]:
# Step 6: Validate Initiatives — parallel LLM calls via DSPy ParallelExecutor
from dspy.utils.parallelizer import ParallelExecutor
from utils.initiative_validator import InitiativeValidator, build_initiative_context

init_validator = InitiativeValidator()
all_df_initiatives = {}  # city_key -> pd.DataFrame

for entry in BATCH:
    key = city_key(entry["metadata"])
    initiative_clusters = [
        c for c in all_clusters[key] if c["cluster_type"] == "parent_with_subs"
    ]
    if not initiative_clusters:
        print(f"No parent_with_subs clusters — skipping.")
        all_df_initiatives[key] = pd.DataFrame()
        continue

    # Per-cluster task 
    def validate_initiative(cluster):
        context = build_initiative_context(cluster)
        prediction = init_validator(initiative_data=context)
        metrics = prediction.initiative_metrics
        return {
            "initiative_name":              context["initiative_name"],
            "parent_statement":             context["parent_statement"],
            "sector":                       context["sector"],
            "num_sub_actions":              context["num_sub_actions"],
            "coverage_score":               metrics.coverage_score,
            "coverage_assessment":          metrics.coverage_assessment,
            "coherence_score":              metrics.coherence_score,
            "coherence_assessment":         metrics.coherence_assessment,
            "aggregate_measurability":      metrics.aggregate_measurability,
            "has_implementation_pathway":   metrics.has_implementation_pathway,
            "inherited_binding_mechanism":  metrics.inherited_binding_mechanism,
            "inherited_spatial_scope":      metrics.inherited_spatial_scope,
            "weak_signals":                 metrics.weak_signals,
            "strong_signals":               metrics.strong_signals,
            "initiative_result":            metrics.initiative_result,
            "confidence_score":             metrics.confidence_score,
            "initiative_reasoning":         metrics.initiative_reasoning,
            "final_verdict":                metrics.final_verdict,
            "sub_assessments": [
                sa.model_dump() if hasattr(sa, "model_dump") else sa.dict()
                for sa in metrics.sub_assessments
            ],
            "subs_strong":   sum(1 for sa in metrics.sub_assessments if sa.strength == "strong"),
            "subs_moderate": sum(1 for sa in metrics.sub_assessments if sa.strength == "moderate"),
            "subs_weak":     sum(1 for sa in metrics.sub_assessments if sa.strength == "weak"),
        }

    # ── Execute in parallel ───────────────────────────────────────────────
    executor = ParallelExecutor(
        num_threads=NUM_THREADS,
        max_errors=len(initiative_clusters),
        provide_traceback=True,
    )
    raw_results = executor.execute(validate_initiative, initiative_clusters)

    # ── Filter failures ───────────────────────────────────────────────────
    results, n_errors = [], 0
    for i, res in enumerate(raw_results):
        if res is None or isinstance(res, Exception):
            n_errors += 1
            print(f"Cluster {i} failed: {res}")
        else:
            results.append(res)

    print(f"[{key}] {len(results)} succeeded, {n_errors} failed")

    # ── Build DataFrame + print summary ──────────────────────────────────
    df_inits = pd.DataFrame(results)
    if not df_inits.empty:
        print(f"  SOUND:   {(df_inits['initiative_result'] == 'SOUND').sum()}")
        print(f"  PARTIAL: {(df_inits['initiative_result'] == 'PARTIAL').sum()}")
        print(f"  WEAK:    {(df_inits['initiative_result'] == 'WEAK').sum()}")
        print(f"  final_verdict=True: {df_inits['final_verdict'].sum()}")

    all_df_initiatives[key] = df_inits

    #Write once, after all results are collected. initiatives are built 
    out = OUTPUT_DIR / f"{key}_validation_initiatives.csv"
    df_inits.to_csv(out, index=False)
    print(f"[{key}] Saved → {out}")


Processed 14 / 14 examples: 100%|██████████| 14/14 [00:46<00:00,  3.31s/it]
[Chicago] 14 succeeded, 0 failed
  SOUND:   8
  PARTIAL: 5
  WEAK:    1
  final_verdict=True: 13
[Chicago] Saved → outputs/Chicago_validation_initiatives.csv
Processed 1 / 1 examples: 100%|██████████| 1/1 [00:00<00:00, 84.43it/s]
[Seattle] 1 succeeded, 0 failed
  SOUND:   0
  PARTIAL: 0
  WEAK:    1
  final_verdict=True: 0
[Seattle] Saved → outputs/Seattle_validation_initiatives.csv
Processed 4 / 4 examples: 100%|██████████| 4/4 [00:00<00:00, 114.56it/s]
[Las_Vegas] 4 succeeded, 0 failed
  SOUND:   0
  PARTIAL: 3
  WEAK:    1
  final_verdict=True: 3
[Las_Vegas] Saved → outputs/Las_Vegas_validation_initiatives.csv
Processed 7 / 7 examples: 100%|██████████| 7/7 [00:00<00:00, 126.18it/s]
[Miami_Dade] 7 succeeded, 0 failed
  SOUND:   6
  PARTIAL: 1
  WEAK:    0
  final_verdict=True: 7
[Miami_Dade] Saved → outputs/Miami_Dade_validation_initiatives.csv
Processed 18 / 18 examples: 100%|██████████| 18/18 [00:52<00:00, 

### Step 7 — Export Combined Results

Builds the final combined table per city and writes `combined_policies.csv`, `trace_individual_policies.csv`, and `trace_initiative_policies.csv` under `outputs/{City_Country}/`.

In [16]:
# Step 7: Export Combined Results per City
from utils.exports import build_combined_policies_table, export_combined_table_and_traces
from utils.exports import filter_valid_policies

all_combined = {}  # city_key -> pd.DataFrame (all policies)
all_valid    = {}  # city_key -> pd.DataFrame (final_verdict == True only)

for entry in BATCH:
    key = city_key(entry["metadata"])
    city_output_dir = OUTPUT_DIR / key
    df_inits = all_df_initiatives.get(key, pd.DataFrame())
    df_inits_arg = df_inits if not df_inits.empty else None

    combined = build_combined_policies_table(
        df_policies=all_df_policies[key],
        df_final_individual=all_df_final[key],
        df_initiatives=df_inits,
        policy_clusters=all_clusters[key],
    )
    all_combined[key] = combined

    # combined_policies.csv now contains valid policies only (individual + initiative clusters)
    # traces contain everything (all rows, full validation detail)
    paths = export_combined_table_and_traces(
        combined=combined,
        df_initiatives=df_inits_arg,
        df_final_individual=all_df_final[key],
        output_dir=city_output_dir,
    )

    # Keep an in-memory copy of valid policies (final_verdict == True)
    df_valid = filter_valid_policies(combined, all_df_final[key], df_inits_arg)
    all_valid[key] = df_valid

    n_all   = len(combined)
    n_valid = len(df_valid)
    print(f"\n[{key}] {n_valid}/{n_all} valid policies  →  {city_output_dir}/combined_policies.csv")
    for k, v in paths.items():
        print(f"  {k}: {v}")



[Chicago] 62/65 valid policies  →  outputs/Chicago/combined_policies.csv
  combined_policies: outputs/Chicago/combined_policies.csv
  valid_policies: outputs/Chicago/valid_policies.csv
  trace_individual: outputs/Chicago/trace_individual_policies.csv
  trace_individual_valid: outputs/Chicago/trace_individual_policies_valid.csv
  trace_initiatives: outputs/Chicago/trace_initiative_policies.csv
  trace_initiatives_valid: outputs/Chicago/trace_initiative_policies_valid.csv

[Seattle] 0/1 valid policies  →  outputs/Seattle/combined_policies.csv
  combined_policies: outputs/Seattle/combined_policies.csv
  valid_policies: outputs/Seattle/valid_policies.csv
  trace_individual: outputs/Seattle/trace_individual_policies.csv
  trace_individual_valid: outputs/Seattle/trace_individual_policies_valid.csv
  trace_initiatives: outputs/Seattle/trace_initiative_policies.csv
  trace_initiatives_valid: outputs/Seattle/trace_initiative_policies_valid.csv

[Las_Vegas] 12/15 valid policies  →  outputs/Las_